In [ ]:
setwd("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/bulk/GTEx/frontal_cortex")

library(dplyr)

source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/marker_vs_mod_corr.R") # CHANGE NAME TO BE ABOUT ENRICHMENT/FGSEA
source("/mnt/lareaulab/reliscu/projects/NSF_GRFP/analyses/code/module_projection_fxns.R")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Loading required package: dynamicTreeCut

Loading required package: fastcluster


Attaching package: ‘fastcluster’


The following object is masked from ‘package:stats’:

    hclust





Attaching package: ‘WGCNA’


The following object is masked from ‘package:stats’:

    cor



Attaching package: ‘qvalue’


The following object is masked from ‘package:WGCNA’:

    qvalue


Loading required package: future


Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths


The following objects are masked from ‘package:data.table’:

    dcast, melt




In [5]:
# Trying another approach to get top cell type modules

In [ ]:
summary_type <- "PC1"

set_source <- "Curated"

marker_genes <- readRDS("/mnt/lareaulab/reliscu/projects/NSF_GRFP/data/marker_genes/AI/Claude_cortical_markers_human.RDS")

In [ ]:
network_dir <- "GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed_mergeParam0.93_subsetCutoff68.086_Modules"
bulk_expr <- fread("GTEx_frontal_cortex_counts_TMMF_SampleNetworks/All_10-58-02/GTEx_frontal_cortex_counts_TMMF_All_200_outliers_removed.csv", data.table=FALSE)
colnames(bulk_expr)[1] <- "Gene"

In [ ]:
# Make sure order of samples matches FM output

networks <- list.files(network_dir, pattern="signum", full.names=TRUE)
networks <- networks[lengths(lapply(networks, list.files)) > 0]
modEigs <- fread(list.files(networks[1], pattern="eigen", full.names=TRUE), data.table=FALSE)
bulk_expr <- bulk_expr[,c(1, match(modEigs$Sample, colnames(bulk_expr)))]

all.equal(modEigs$Sample, colnames(bulk_expr)[-1])

[1] TRUE

In [ ]:
# Restrict module space:

# Find module that is most correlated with PC1 of marker genes
# Run FGSEA on kME vector for that module

marker_genes <- lapply(marker_genes, function(markers) {
    markers[markers %in% bulk_expr[,1]]
})

ct_mod_corrs <- marker_vs_mod_corr(bulk_expr, networks,  marker_genes, summary_type)

In [ ]:
top_mods_df <- ct_mod_corrs %>%
    group_by(Cell_type) %>%
    slice_max(Corr) %>%
    slice_min(padj) %>%
    arrange(padj) %>%
    filter(padj < .05)

In [ ]:
fwrite(top_mods_df, file=paste0("data/FGSEA/", network_dir, "_Curated_", summary_type, "_FGSEA_top_Qval_mods.csv"), row.names=FALSE)

#### Check for conflicting enrichments in top cell type modules:

In [ ]:
ct <- "VLMC"
network = top_mods_df$Network[top_mods_df$Cell_type == ct] 
mod = top_mods_df$Mod[top_mods_df$Cell_type == ct]

mask <- ct_mod_corrs$Network == network & ct_mod_corrs$Mod == mod
ct_mod_corrs[mask,] %>%
    filter(padj < .05)

In [ ]:
ct <- "All Neuronal"
network = top_mods_df$Network[top_mods_df$Cell_type == ct] 
mod = top_mods_df$Mod[top_mods_df$Cell_type == ct]

mask <- ct_mod_corrs$Network == network & ct_mod_corrs$Mod == mod
ct_mod_corrs[mask,] %>%
    filter(padj < .05)

In [ ]:
ct <- "All Glutamatergic"
network = top_mods_df$Network[top_mods_df$Cell_type == ct] 
mod = top_mods_df$Mod[top_mods_df$Cell_type == ct]

mask <- ct_mod_corrs$Network == network & ct_mod_corrs$Mod == mod
ct_mod_corrs[mask,] %>%
    filter(padj < .05)

,Cell_type,Network,Mod,Corr,Top_kME_genes,ME_path,pathway,pval,padj,log2err,ES,NES,size,leadingEdge
,<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
blue,All Neuronal,Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747,blue,0.9330768,"ENO2, KCNC1, SNHG14, SLC17A7, TUBGCP5, SATB2, THY1, TNFRSF11A, GLCE, CRYM, EMX1, RBFOX3, CPNE5, VLDLR, INCENP",GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules/Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747/Module_eigengenes_09-14-33.csv,All Neuronal,7.814300e-09,7.814300e-09,0.7477397,0.9696423,1.910912,9,"ENO2, THY1, RBFOX3, NRGN, STMN2, TUBB3, SYT1, SNAP25, MAP2"
blue6,All Glutamatergic,Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747,blue,0.9384606,"ENO2, KCNC1, SNHG14, SLC17A7, TUBGCP5, SATB2, THY1, TNFRSF11A, GLCE, CRYM, EMX1, RBFOX3, CPNE5, VLDLR, INCENP",GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules/Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747/Module_eigengenes_09-14-33.csv,All Glutamatergic,9.805086e-08,9.805086e-08,0.7049757,0.9657588,1.852909,8,"SLC17A7, SATB2, EMX1, TBR1, CAMK2A, NEUROD6, NEUROD2, SLC17A6"
blue7,L4 IT,Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747,blue,0.8542282,"ENO2, KCNC1, SNHG14, SLC17A7, TUBGCP5, SATB2, THY1, TNFRSF11A, GLCE, CRYM, EMX1, RBFOX3, CPNE5, VLDLR, INCENP",GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules/Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747/Module_eigengenes_09-14-33.csv,L4 IT,2.688505e-02,2.688505e-02,0.3524879,0.8130344,1.554303,6,"COL5A2, CTXN3, RORB, THSD7A, RSPO1"
blue8,Chandelier,Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747,blue,0.9315038,"ENO2, KCNC1, SNHG14, SLC17A7, TUBGCP5, SATB2, THY1, TNFRSF11A, GLCE, CRYM, EMX1, RBFOX3, CPNE5, VLDLR, INCENP",GTEx_cortex_counts_TMMF_All_501_outliers_removed_47840genes_cleaned_44846genes_cleaned_mergeParam0.85_subsetCutoff1.427_Modules/Bicor-None_signum0.349_minSize10_merge_ME_0.85_19747/Module_eigengenes_09-14-33.csv,Chandelier,3.796204e-02,3.796204e-02,0.2311267,0.7912101,1.516912,6,"SNHG14, CPNE5, PVALB, VIPR2, UNC5B, NOS1"


In [ ]:
ct <- "L2/3 IT"
network = top_mods_df$Network[top_mods_df$Cell_type == ct] 
mod = top_mods_df$Mod[top_mods_df$Cell_type == ct]

mask <- ct_mod_corrs$Network == network & ct_mod_corrs$Mod == mod
ct_mod_corrs[mask,] %>%
    filter(padj < .05)

#### Save MEs

In [ ]:
modEig_list <- lapply(1:nrow(top_mods_df), function(i) {
    modEig <- fread(top_mods_df$ME_path[i], data.table=FALSE)
    col_idx <- which(colnames(modEig) == top_mods_df$Mod[i]) 
    modEig <- modEig[,c(1, col_idx)]
    colnames(modEig)[2] <- top_mods_df$Cell_type[i]
    modEig 
})

modEig_df <- Reduce(function(x, y) merge(x, y, by="Sample"), modEig_list)

In [ ]:
fwrite(top_mods_df, file=paste0("data/FGSEA/", network_dir, "_Curated_", summary_type, "_FGSEA_top_Qval_mods_eigenges.csv"), row.names=FALSE)